In [ ]:
import ipywidgets as W
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Javascript, display

from brand_kit.theme import DIVERGING, FONT, GRAY, GRID, NAVY, NAVY_RAMP, ORANGE, TEXT, TEXT_SOFT, WHITE

df = pd.read_csv("ai_job_dataset.csv", parse_dates=["posting_date"])
df["month"] = df["posting_date"].dt.to_period("M").dt.to_timestamp()

LEVELS = ["Entry", "Mid", "Senior", "Executive"]
MODES = ["On-site", "Hybrid", "Fully remote"]
SIZES = ["Small", "Medium", "Large"]
df["experience"] = df["experience_level"].map(dict(zip(["EN", "MI", "SE", "EX"], LEVELS)))
df["work_mode"] = df["remote_ratio"].map(dict(zip([0, 50, 100], MODES)))
df["size"] = df["company_size"].map(dict(zip("SML", SIZES)))
skills = df["required_skills"].str.get_dummies(sep=", ")
MONTHS = sorted(df["month"].unique())
COUNTRIES = sorted(df["company_location"].unique())
MIN_N = 30  # smallest group we will draw a median for

signed = lambda v: f"{'−' if v < 0 else '+'}${abs(v):,.0f}"
money = lambda v: f"${v / 1000:,.0f}k"

iso = lambda idx: [f"{d:%Y-%m-%d}" for d in idx]  # plain dates: no timezone shift in the browser

def r_squared(y, X):
    """Adjusted R2: share of variation in y explained by dummy columns X (0-1)."""
    X = np.column_stack([np.ones(len(y)), X])
    resid = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
    k = np.linalg.matrix_rank(X)
    return max(0.0, 1 - (resid @ resid / (len(y) - k)) / y.var(ddof=1))

def skill_premium(sub):
    """Median pay with a skill minus without it; NaN where either side is too small."""
    sk, pay = skills.loc[sub.index], sub["salary_usd"]
    out = {}
    for s in sk:
        a, b = pay[sk[s] == 1], pay[sk[s] == 0]
        out[s] = a.median() - b.median() if min(len(a), len(b)) >= MIN_N else np.nan
    return pd.Series(out)

assert len(df) == 15000 and not df[["experience", "work_mode", "size"]].isna().any().any()
_g = np.repeat([0, 1], 50)
assert r_squared(_g * 10.0, _g) > 0.999 and r_squared(np.tile([0.0, 1.0], 50), _g) == 0

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Libre+Franklin:wght@400;600;800&display=swap');
:root { --navy:#232D4B; --gray:#C9CBD2; --orange:#E57200; --peach:#F9DCBE; --soft:#4A5677; --page:#F6F6F8; --line:#E4E5EA;
  --jp-widgets-slider-active-handle-color:var(--navy); --jp-widgets-input-focus-border-color:var(--orange); }
body, #rendered_cells { background:var(--page) !important; }
#rendered_cells { padding:0 !important; }
#rendered_cells .jp-Cell { padding:0 20px !important; margin:0 auto !important; max-width:1720px; box-sizing:border-box; }
#rendered_cells .jp-Cell:has(.band) { padding:0 !important; max-width:none; }
#rendered_cells .jp-OutputArea-output { padding:0 !important; background:transparent; }
body, .jp-OutputArea-output, .jp-RenderedHTMLCommon, .widget-label, .widget-readout, .widget-button, .widget-toggle-button, .widget-dropdown select
  { font-family:'Libre Franklin','Franklin Gothic',Arial,sans-serif !important; }
.jp-RenderedHTMLCommon { color:var(--navy) !important; font-size:15px !important; line-height:1.5 !important; padding:0 !important; margin:0 !important; }
.jp-OutputArea-child { table-layout:fixed !important; width:100% !important; }
.jp-OutputPrompt, .jp-InputPrompt { display:none !important; }
.band { background:var(--navy); border-bottom:8px solid var(--orange); padding:30px max(20px, calc((100vw - 1680px) / 2)) 26px; }
.band h1 { color:#fff !important; font-size:32px !important; line-height:1.15 !important; font-weight:800 !important; margin:0 0 8px !important; }
.band p { color:var(--peach) !important; margin:0 !important; max-width:80ch; }
.band p + p { font-size:13px; margin-top:8px !important; }
.sec { font-size:24px !important; line-height:1.2 !important; font-weight:800 !important; color:var(--navy) !important; margin:32px 0 4px !important; }
.lede { margin:0 0 12px !important; max-width:80ch; }
.intro { display:grid; grid-template-columns:1.25fr 1fr; gap:12px; margin-top:20px; }
.card, .panel, .filters, .tile, .insights figure { background:#fff; border:1px solid var(--line); border-radius:8px; box-sizing:border-box; }
.card { padding:16px 20px; }
.card h2, .card h2:first-child { font-size:18px !important; font-weight:800 !important; color:var(--navy) !important; margin:0 0 8px !important; }
.card ul, .card ol { margin:0 !important; padding-left:20px !important; }
.card li { margin-bottom:5px !important; }
.card a { color:var(--navy) !important; text-decoration:underline !important; text-decoration-color:var(--orange); text-underline-offset:3px; }
.flag { background:var(--peach); border-radius:6px; padding:8px 12px; margin:12px 0 0 !important; font-size:14px; }
.insights { display:grid; grid-template-columns:repeat(auto-fit,minmax(290px,1fr)); gap:12px; }
.insights figure { margin:0; padding:16px 20px; }
.insights figcaption { margin-bottom:12px; min-height:3em; }
.insights figcaption b { display:block; font-size:18px; font-weight:800; }
.hb { display:grid; grid-template-columns:104px 1fr; align-items:center; gap:8px; font-size:13px; margin:6px 0; }
.hb > span:last-child { display:flex; align-items:center; gap:8px; font-weight:600; padding-right:48px; }
.hb i { display:block; flex:none; height:20px; border-radius:0 4px 4px 0; min-width:2px; }
.root > * { margin:0 0 12px !important; }
.filters { padding:8px 16px; flex-wrap:wrap !important; gap:6px 20px; align-items:center; }
.tiles { display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:12px; }
.tile { padding:12px 16px; }
.tile b { display:block; font-size:24px; line-height:1.2; color:var(--navy); font-weight:800; }
.tile span { display:block; line-height:1.35; font-size:13px; color:var(--soft); }
.dash { display:grid !important; gap:12px; align-items:stretch; grid-template-columns:1.2fr 1.2fr 1.3fr .95fr; width:100%; overflow:visible !important; }
.dash > *, .col > * { min-width:0; }
.col { display:flex; flex-direction:column; gap:12px; margin:0 !important; overflow:visible !important; }
.panel { padding:14px 16px; margin:0 !important; overflow:hidden; }
.grow { flex:1 1 auto; }
.panel h3 { font-size:16px !important; line-height:1.3 !important; font-weight:800 !important; margin:0 !important; color:var(--navy) !important; }
.panel p { font-size:13px; line-height:1.4; margin:4px 0 2px !important; color:var(--soft); }
.panel .widget-label, .panel .widget-readout { font-size:13px; color:var(--soft); }
.widget-dropdown select { color:var(--navy) !important; border:1px solid var(--gray) !important; border-radius:6px; background-color:#fff !important; }
.widget-toggle-button, .widget-button { background:var(--line) !important; color:var(--navy) !important; border-radius:6px; box-shadow:none !important; font-weight:600; }
.widget-toggle-button.mod-active { background:var(--navy) !important; color:#fff !important; }
.widget-button:focus-visible, .widget-toggle-button:focus-visible, .widget-dropdown select:focus-visible { outline:3px solid var(--orange) !important; outline-offset:1px; }
.noUi-connect { background:var(--navy) !important; }
.noUi-handle { border-color:var(--navy) !important; }
.answer { background:var(--navy) !important; border-color:var(--navy) !important; }
.answer .big { font-size:60px; line-height:1; font-weight:800; color:#fff; margin:6px 0 4px; }
.answer h3 { color:var(--peach) !important; }
.answer p { color:#fff; font-size:15px; line-height:1.5; margin:0 0 12px !important; }
.answer p.fine { color:var(--peach); font-size:13px; }
.foot { color:var(--soft); font-size:13px; margin:4px 0 24px !important; }
@media (max-width:1150px) { .dash { grid-template-columns:1fr 1fr; } }
@media (max-width:760px) { .dash, .intro { grid-template-columns:1fr; } }
</style>
<div class="band"><h1>What does the AI job market reward most?</h1>
<p>Our goal: help job seekers see which parts of a job most affect pay: experience, location, company size, role, skills, or work arrangement.</p>
<p>By #unemployed: Michael, Brandon, Mia, Hassan, Jafar</p></div>"""))

In [ ]:
display(HTML("""
<div class="intro">
<section class="card"><h2>About the data</h2><ul>
<li><b>Source:</b> <a href="https://www.kaggle.com/datasets/bismasajjad/global-ai-job-market-and-salary-trends-2025" target="_blank" rel="noopener">Global AI Job Market &amp; Salary Trends 2025</a> by Bisma Sajjad on Kaggle (CC0 public domain).</li>
<li><b>Contents:</b> 15,000 job postings, one per row, from January 2024 to April 2025. They cover 20 job titles and 20 countries.</li>
<li><b>Key variables:</b> salary (USD), experience level, job title, company location, company size, employment type, remote ratio, required skills, and posting date.</li></ul>
<p class="flag"><b>Data is synthetic.</b> Patterns are not real salary advice. The author generated the postings by algorithm for educational use.</p></section>
<section class="card"><h2>Research questions</h2><ol>
<li><b>Experience:</b> How does pay change as people gain experience?</li>
<li><b>Work location:</b> Which countries pay AI workers the most?</li>
<li><b>Skills:</b> Do jobs that need a particular skill pay more?</li>
<li><b>Over time:</b> Is hiring growing or shrinking month to month?</li>
<li><b>Over time:</b> Have any roles had unusual pay spikes or drops?</li>
<li><b>The answer:</b> What does the market reward most?</li></ol></section>
</div>
<h2 class="sec">Key insights</h2>"""))

In [ ]:
def static_bars(s, fmt, title, caption):
    """Static bar chart; the first-ranked value is orange."""
    rows = "".join(f'<div class="hb"><span>{k}</span><span><i style="width:{v / s.max():.1%};background:{ORANGE if v == s.max() else GRAY}"></i>{fmt(v)}</span></div>'
                   for k, v in s.items())
    return f"<figure><figcaption><b>{title}</b>{caption}</figcaption>{rows}</figure>"

pay = df["salary_usd"].to_numpy(float)
dummies = lambda c: pd.get_dummies(df[c]).to_numpy(float)
by_level = df.groupby("experience")["salary_usd"].median()[LEVELS]
by_place = df.groupby("company_location")["salary_usd"].median().sort_values(ascending=False)
ends = pd.concat([by_place.head(3), by_place.tail(3)])
explained = pd.Series({"Experience": r_squared(pay, dummies("experience")),
                       "Country": r_squared(pay, dummies("company_location")),
                       "Company size": r_squared(pay, dummies("size")),
                       "Job title": r_squared(pay, dummies("job_title")),
                       "Skills": r_squared(pay, skills.to_numpy(float)),
                       "Work mode": r_squared(pay, dummies("work_mode"))}).sort_values(ascending=False)

display(HTML('<div class="insights">' + "".join([
    static_bars(by_level, money, "Experience pays", f"The median executive posting pays {by_level['Executive']/by_level['Entry']:.1f} times the median entry-level posting."),
    static_bars(ends, money, "Location comes second", f"{by_place.index[0]}'s median is {by_place.iloc[0]/by_place.iloc[-1]:.1f} times {by_place.index[-1]}'s. Top three and bottom three countries shown."),
    static_bars(explained, lambda v: "<1%" if v < .01 else f"{v:.0%}", "Experience explains the most pay variation",
                "Job title, skills and work mode each explain less than 1%."),
]) + "</div>"))
assert list(explained.index[:3]) == ["Experience", "Country", "Company size"] and explained.iloc[3:].max() < .01  # captions above stay true

In [ ]:
display(HTML("""<h2 class="sec">Explore the data</h2>
<p class="lede">The two filters below update the headline numbers and every chart that can use them. Each chart also has its own controls. Click a bar in the country chart to filter by that country, and click it again to clear the filter.</p>"""))

In [ ]:
# ---------- shared filters ----------
wide = W.Layout(width="auto", min_width="0")
f_level = W.Dropdown(options=["All levels"] + LEVELS, description="Experience", layout=W.Layout(width="230px"))
f_country = W.Dropdown(options=["All countries"] + COUNTRIES, description="Country", layout=W.Layout(width="250px"))
f_reset = W.Button(description="Reset filters", layout=W.Layout(width="110px"))
tiles = W.HTML()

def subset(level=True, country=True):
    m = pd.Series(True, index=df.index)
    if level and f_level.value != "All levels": m &= df["experience"] == f_level.value
    if country and f_country.value != "All countries": m &= df["company_location"] == f_country.value
    return df[m]

def scope(level=True, country=True):
    parts = [f_level.value.lower() if level and f_level.value != "All levels" else "all levels",
             f_country.value if country and f_country.value != "All countries" else "all countries"]
    return ", ".join(parts)

# ---------- chart scaffolding ----------
def make_fig(height):
    f = go.Figure()
    f.update_layout(template="none", height=height, autosize=True, margin=dict(l=8, r=8, t=8, b=8), showlegend=False,
                    font=dict(family=FONT, size=12, color=TEXT),
                    paper_bgcolor=WHITE, plot_bgcolor=WHITE, hoverlabel=dict(bgcolor="white", font_size=13),
                    dragmode=False, bargap=0.3)
    f.update_xaxes(gridcolor=GRID, zeroline=False, automargin=True, fixedrange=True)
    f.update_yaxes(gridcolor=GRID, zeroline=False, automargin=True, fixedrange=True)
    return f

def panel(question, controls, grow=False):
    head = W.HTML()
    p = W.VBox([W.HTML(f"<h3>{question}</h3>"), head, *controls])
    p.add_class("panel")
    if grow: p.add_class("grow")
    return p, head

def note(head, finding, sc=""):
    head.value = f"<p><b style='color:{TEXT}'>{finding}</b>{'<br>' + sc if sc else ''}</p>"

def replace(fig, traces):
    with fig.batch_update():
        fig.data = []
        fig.add_traces(traces)

# ---------- 1. experience ----------
SPLITS = {"None": None, "Company size": ("size", SIZES), "Work mode": ("work_mode", MODES)}
SPLIT_COLORS = [NAVY_RAMP[0], NAVY_RAMP[1], NAVY_RAMP[3]]
c_split = W.Dropdown(options=list(SPLITS), description="Split by", layout=wide)
fig1 = make_fig(330)
fig1.update_layout(boxmode="group", legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0))
fig1.update_yaxes(tickprefix="$", tickformat="~s", rangemode="tozero", title="Salary (USD)")
p1, h1 = panel("Experience: How does pay change as people gain experience?", [c_split])

def draw1(*_):
    sub, split = subset(level=False), SPLITS[c_split.value]
    groups = [(None, sub, NAVY)] if split is None else [(g, sub[sub[split[0]] == g], c) for g, c in zip(split[1], SPLIT_COLORS)]
    traces = []
    for name, s, color in groups:
        s = s[s["experience"].isin([l for l, n in s["experience"].value_counts().items() if n >= 10])]
        q = s.groupby("experience")["salary_usd"].quantile([.05, .25, .5, .75, .95]).unstack()
        q = q.reindex([l for l in LEVELS if l in q.index])
        n = s["experience"].value_counts()
        traces.append(go.Box(x=list(q.index), name=name or "All", lowerfence=list(q[.05]), q1=list(q[.25]), median=list(q[.5]),
                             q3=list(q[.75]), upperfence=list(q[.95]), line=dict(color=color, width=2), hoverinfo="y",
                             fillcolor=GRID if split is None else WHITE))
    replace(fig1, traces)
    with fig1.batch_update():
        fig1.update_layout(showlegend=split is not None)
        fig1.update_xaxes(categoryorder="array", categoryarray=LEVELS)
    m = sub.groupby("experience")["salary_usd"].median()
    note(h1, f"Entry {money(m['Entry'])} → Executive {money(m['Executive'])} ({m['Executive']/m['Entry']:.1f}×)",
         f"Box: middle half, line: median, whiskers: 5th to 95th percentile. Shows {scope(level=False)}.")

# ---------- 2. country ----------
c_sort = W.ToggleButtons(options=["Highest", "Lowest", "A–Z"], style=W.ToggleButtonsStyle(button_width="76px"))
fig2 = make_fig(500)
fig2.add_bar(orientation="h", textposition="outside", cliponaxis=False,
             hovertemplate="<b>%{y}</b><br>Median %{x:$,.0f}<br>%{customdata:,} postings<extra></extra>")
fig2.update_xaxes(visible=False)
fig2.update_yaxes(showgrid=False, dtick=1)
fig2.update_layout(margin_r=45)
p2, h2 = panel("Work location: Which countries pay AI workers the most?", [c_sort], grow=True)

def draw2(*_):
    g = subset(country=False).groupby("company_location")["salary_usd"].agg(["median", "size"])
    s = g.sort_index(ascending=False) if c_sort.value == "A–Z" else g.sort_values("median", ascending=c_sort.value == "Highest")
    best, worst = g["median"].idxmax(), g["median"].idxmin()
    hot = f_country.value if f_country.value in s.index else best
    with fig2.batch_update():
        b = fig2.data[0]
        b.x, b.y = list(s["median"]), list(s.index)
        b.text, b.customdata = [money(v) for v in s["median"]], list(s["size"])
        b.marker.color = [ORANGE if c == hot else GRAY for c in s.index]
        fig2.update_xaxes(range=[0, s["median"].max() * 1.05])
        fig2.update_yaxes(categoryorder="array", categoryarray=list(s.index))
    note(h2, f"{best} pays {g['median'][best]/g['median'][worst]:.1f}× as much as {worst}", f"Median salary, {scope(country=False)}. Click a bar to filter.")

def click_country(trace, points, state):
    if points.point_inds:
        c = trace.y[points.point_inds[0]]
        f_country.value = "All countries" if f_country.value == c else c

# ---------- 3. skills ----------
c_meas = W.ToggleButtons(options=["Pay difference", "Demand"], style=W.ToggleButtonsStyle(button_width="112px"))
c_top = W.IntSlider(value=12, min=6, max=24, step=2, description="Skills shown", continuous_update=False, layout=wide,
                    style={"description_width": "80px"})
fig3 = make_fig(400)
fig3.update_yaxes(showgrid=False, dtick=1)
fig3.add_bar(orientation="h", width=0.12, hoverinfo="skip")
fig3.add_scatter(mode="markers", marker_size=10)
p3, h3 = panel("Skills: Do jobs that need a particular skill pay more?", [c_meas, c_top])

def draw3(*_):
    sub = subset()
    k = c_top.value
    demand = skills.loc[sub.index].sum()
    pay_mode = c_meas.value == "Pay difference"
    full = (skill_premium(sub).dropna() if pay_mode else demand / len(sub)).sort_values()
    # pay: both ends of the ranking; demand: the most requested
    s = full if len(full) <= k else (pd.concat([full.head(k // 2), full.tail(k // 2)]) if pay_mode else full.tail(k))
    with fig3.batch_update():
        colors = [NAVY if pay_mode and v > 0 else NAVY_RAMP[1] for v in s.values]
        if len(s): colors[-1] = ORANGE
        for t in fig3.data:
            t.x, t.y, t.marker.color = list(s.values), list(s.index), colors
        fig3.data[1].customdata = list(demand[s.index])
        fig3.data[1].hovertemplate = ("<b>%{y}</b><br>%{x:+$,.0f} vs. jobs without it" if pay_mode else "<b>%{y}</b><br>%{x:.0%} of postings") + "<br>%{customdata:,} postings<extra></extra>"
        lim = max(12000, s.abs().max() * 1.15) if pay_mode and len(s) else 1
        fig3.update_xaxes(range=[-lim, lim] if pay_mode else [0, max(.01, s.max() * 1.15)], nticks=5, zeroline=True, zerolinecolor=TEXT_SOFT,
                          tickformat="$~s" if pay_mode else ".0%", title="Difference in median salary" if pay_mode else "Share of postings")
        fig3.update_yaxes(categoryorder="array", categoryarray=list(s.index))
    if not len(s):
        note(h3, "Too few postings to compare skills", f"Shows {scope()}.")
    elif pay_mode:
        cut = "" if len(full) <= k else f" Showing the {k // 2} highest and {k // 2} lowest."
        small = " Small sample, so expect noise." if len(sub) < 1500 else ""
        note(h3, f"{full.index[-1]} {signed(full.iloc[-1])}, {full.index[0]} {signed(full.iloc[0])}",
             f"Median pay with the skill minus without it, {scope()}.{cut}{small}")
    else:
        note(h3, f"{full.index[-1]} is the most requested ({full.iloc[-1]:.0%})", f"Share of postings listing each skill, {scope()}.")

# ---------- 4. hiring ----------
labels = [(pd.Timestamp(m).strftime("%b %y"), i) for i, m in enumerate(MONTHS)]
c_range = W.SelectionRangeSlider(options=labels, index=(0, len(MONTHS) - 1), description="Months", continuous_update=False, layout=wide,
                                 style={"description_width": "50px"})
fig4 = make_fig(345)
fig4.add_scatter(mode="lines+markers", line=dict(color=NAVY, width=2), marker_size=8,
                 hovertemplate="%{x|%B %Y}<br>%{y:,} postings<extra></extra>")
fig4.add_scatter(mode="lines", line=dict(color=TEXT_SOFT, width=1), hoverinfo="skip")
fig4.update_yaxes(title="Postings", rangemode="tozero")
fig4.update_xaxes(tickformat="%b<br>%Y", nticks=6, showgrid=False)
p4, h4 = panel("Over time: Is hiring growing or shrinking month to month?", [c_range], grow=True)

def draw4(*_):
    lo, hi = c_range.value
    s = subset().groupby("month").size().reindex(MONTHS, fill_value=0).iloc[lo:hi + 1]
    with fig4.batch_update():
        fig4.data[0].x, fig4.data[0].y = iso(s.index), list(s.values)
        fig4.data[1].x, fig4.data[1].y = iso(s.index[[0, -1]]), [s.mean()] * 2
    if len(s) < 2:
        note(h4, f"{s.iloc[0]:,} postings in {s.index[0]:%b %Y}", f"Widen the range to see a trend. Shows {scope()}.")
        return
    change = f" ({s.iloc[-1] / s.iloc[0] - 1:+.0%})" if s.iloc[0] else ""
    note(h4, f"{s.iloc[0]:,} → {s.iloc[-1]:,} postings{change}", f"{s.index[0]:%b %Y} to {s.index[-1]:%b %Y}; gray line is the average, {s.mean():,.0f} a month. Shows {scope()}.")

# ---------- 5. unusual pay ----------
c_sd = W.FloatSlider(value=1.5, min=0, max=3, step=0.25, description="Threshold (SD)", continuous_update=False, layout=wide,
                     style={"description_width": "95px"}, readout_format=".2f")
c_role = W.Dropdown(options=["All roles"] + sorted(df["job_title"].unique()), description="Role", layout=wide, style={"description_width": "40px"})
tbl = df.pivot_table(index="job_title", columns="month", values="salary_usd", aggfunc="median")
pct = tbl.div(tbl.median(axis=1), axis=0) - 1
zs = tbl.sub(tbl.mean(axis=1), axis=0).div(tbl.std(axis=1), axis=0)
order = zs.abs().max(axis=1).sort_values().index  # most unusual role on top
tbl, pct, zs = tbl.loc[order], pct.loc[order], zs.loc[order]
fig5 = make_fig(940)
fig5.add_heatmap(x=iso(tbl.columns), y=list(order), zmid=0, zmin=-.3, zmax=.3, xgap=2, ygap=2,
                 colorscale=DIVERGING,
                 colorbar=dict(title=dict(text="Pay vs. the role's usual level", side="top"), tickformat="+.0%", orientation="h", thickness=8, y=-.07, yanchor="top", len=1),
                 hovertemplate="<b>%{y}</b>, %{x|%b %Y}<br>Median %{customdata[0]:$,.0f}<br>%{customdata[1]:+.0%} vs. usual (%{customdata[2]:+.1f} SD)<extra></extra>",
                 customdata=np.dstack([tbl, pct, zs]).tolist())
fig5.add_scatter(mode="lines+markers", line=dict(color=NAVY, width=2), visible=False, xaxis="x", yaxis="y2",
                 marker=dict(size=10), hovertemplate="%{x|%b %Y}<br>Median %{y:$,.0f}<extra></extra>")
fig5.update_layout(yaxis2=dict(overlaying="y", tickmode="auto", nticks=8, rangemode="tozero", visible=False, tickprefix="$", tickformat="~s", automargin=True, gridcolor=GRID, title="Median salary (USD)", fixedrange=True, anchor="x"))
fig5.update_xaxes(tickformat="%b<br>%Y", nticks=5, showgrid=False)
fig5.update_yaxes(showgrid=False, dtick=1, tickfont_size=11, selector=dict(anchor=None))
p5, h5 = panel("Over time: Have any roles had unusual pay spikes or drops?", [c_role, c_sd], grow=True)

def draw5(*_):
    flag, role = zs.abs() >= c_sd.value, c_role.value
    one = role != "All roles"
    with fig5.batch_update():
        fig5.data[0].visible, fig5.data[1].visible = not one, one
        fig5.update_layout(yaxis_visible=not one, yaxis2_visible=one)
        if one:
            hits = flag.loc[role]
            fig5.data[1].x, fig5.data[1].y = iso(tbl.columns), list(tbl.loc[role])
            fig5.data[1].marker.color = [NAVY if v and h else ORANGE if h else NAVY_RAMP[0] for h, v in zip(hits, zs.loc[role] > 0)]
        else:
            fig5.data[0].z = pct.where(flag, 0).to_numpy().tolist()  # lists: 2-D numpy arrays render blank in Voila
    if one:
        note(h5, f"{int(hits.sum())} of {len(hits)} months are unusual for {role}", "Navy dots are spikes and orange dots are drops. Covers all levels and countries.")
    else:
        stacked = zs.stack()
        r, m = stacked.abs().idxmax()
        note(h5, f"{int(flag.sum().sum())} of {zs.size} role-months are unusual. Most unusual: {r}, {m:%b %Y} ({pct.loc[r, m]:+.0%})",
             "Colored squares pass the slider's threshold. Covers all levels and countries; filtered samples would be too small.")

# ---------- 6. the answer ----------
c_metric = W.ToggleButtons(options=["Pay gap", "% explained"], style=W.ToggleButtonsStyle(button_width="96px"))
FACTORS = {"Experience": "experience", "Country": "company_location", "Company size": "size", "Job title": "job_title", "Skills": None, "Work mode": "work_mode"}
fig6 = make_fig(560)
fig6.add_bar(orientation="h", textposition="outside", cliponaxis=False, hovertemplate="<b>%{y}</b><br>%{text}<br>%{customdata}<extra></extra>")
fig6.update_xaxes(visible=False)
fig6.update_yaxes(showgrid=False, autorange="reversed")
fig6.update_layout(margin_r=40)
p6, h6 = panel("The answer: How much does typical pay change between the best- and worst-paying group for each factor?", [c_metric])
takeaway = W.HTML(f"""<h3>Takeaway</h3><div class="big">{explained["Experience"]:.0%}</div>
<p>of pay variation is explained by experience alone, across all postings.</p>
<p>Country comes second at {explained["Country"]:.0%}, then company size at {explained["Company size"]:.0%}. Job title, skills and work mode each explain under 1%.</p>
<p class="fine">Data is synthetic. Patterns are not real salary advice.</p>""")
p7 = W.VBox([takeaway]); p7.add_class("panel"); p7.add_class("answer"); p7.add_class("grow")

def draw6(*_):
    sub, gap_mode = subset(), c_metric.value == "Pay gap"
    vals, detail = {}, {}
    for name, col in FACTORS.items():
        if col and sub[col].nunique() < 2: continue  # held fixed by a filter
        if gap_mode:
            if col:
                g = sub.groupby(col)["salary_usd"].agg(["median", "size"])
                g = g.loc[g["size"] >= 10, "median"]
            else:
                g = skill_premium(sub).dropna()
            if len(g) < 2: continue
            vals[name], detail[name] = g.max() - g.min(), f"{g.idxmax()} vs. {g.idxmin()}"
        else:
            X = pd.get_dummies(sub[col]).to_numpy(float) if col else skills.loc[sub.index].to_numpy(float)
            vals[name], detail[name] = r_squared(sub["salary_usd"].to_numpy(float), X), "of the variation in salary (adjusted R²)"
    s = pd.Series(vals).sort_values(ascending=False)
    fmt = money if gap_mode else (lambda v: "<1%" if v < .01 else f"{v:.0%}")
    with fig6.batch_update():
        b = fig6.data[0]
        b.y, b.x, b.text, b.customdata = list(s.index), list(s.values), [fmt(v) for v in s.values], [detail[i] for i in s.index]
        b.marker.color = [ORANGE] + [GRAY] * (len(s) - 1)
        fig6.update_xaxes(range=[0, max(s.max() * 1.1, .01)])
    fixed = " Factors you filter on are held fixed and drop out." if len(s) < len(FACTORS) else ""
    small = " Small groups inflate gaps; check % explained." if gap_mode and len(sub) < 1500 else ""
    if gap_mode:
        note(h6, f"{s.index[0]} has the largest gap: {fmt(s.iloc[0])}", f"Best minus worst group median, {scope()}. For skills, best minus worst pay difference.{fixed}{small}")
    else:
        note(h6, f"{s.index[0]} explains the most: {fmt(s.iloc[0])}", f"Share of salary variation each factor explains by itself, {scope()}.{fixed}")

# ---------- headline numbers ----------
def draw_tiles(*_):
    sub = subset()
    d = skills.loc[sub.index].mean()
    t = sub.groupby("job_title")["salary_usd"].median()
    facts = [(money(sub["salary_usd"].median()), "median salary"), (f"{len(sub):,}", "job postings"),
             (f"{d.idxmax()} ({d.max():.0%})", "most requested skill"), (f"{(sub['remote_ratio'] == 100).mean():.0%}", "fully remote"),
             (t.idxmax(), f"top-paying title ({money(t.max())} median)")]
    tiles.value = '<div class="tiles">' + "".join(f'<div class="tile"><b>{a}</b><span>{b}</span></div>' for a, b in facts) + "</div>"

# ---------- wiring ----------
def on(widgets, fn):
    for w in widgets: w.observe(fn, names="value")

on([f_country, c_split], draw1)
on([f_level, f_country, c_sort], draw2)
on([f_level, f_country, c_meas, c_top], draw3)
on([f_level, f_country, c_range], draw4)
on([c_sd, c_role], draw5)
on([f_level, f_country, c_metric], draw6)
on([f_level, f_country], draw_tiles)

def reset(_):
    f_level.value, f_country.value = "All levels", "All countries"
f_reset.on_click(reset)

for fn in (draw1, draw2, draw3, draw4, draw5, draw6, draw_tiles): fn()

# Convert to live widgets only now: Voila's front end sees a widget's initial state,
# so figures changed after becoming a FigureWidget but before display render blank.
fig1, fig2, fig3, fig4, fig5, fig6 = [go.FigureWidget(f) for f in (fig1, fig2, fig3, fig4, fig5, fig6)]
for p, f in zip((p1, p2, p3, p4, p5, p6), (fig1, fig2, fig3, fig4, fig5, fig6)):
    f._config = {**f._config, "displayModeBar": False, "responsive": True}
    p.children += (f,)
fig2.data[0].on_click(click_country)

def col(*panels):
    c = W.VBox(panels); c.add_class("col"); return c
filters = W.HBox([f_level, f_country, f_reset]); filters.add_class("filters")
grid = W.Box([col(p1, p2), col(p3, p4), col(p5), col(p6, p7)]); grid.add_class("dash")
root = W.VBox([filters, tiles, grid]); root.add_class("root")
display(root)
# Charts can draw before the page layout settles and stick at Plotly's default 700px.
# Nudge Plotly until every chart fits its card (gives up after 60s).
display(Javascript("""
let tries = 0;
const timer = setInterval(() => {
  const plots = [...document.querySelectorAll('.js-plotly-plot')];
  if (plots.some(p => p._fullLayout && Math.abs(p._fullLayout.width - p.clientWidth) > 1)) window.dispatchEvent(new Event('resize'));
  if (++tries > 120) clearInterval(timer);
}, 500);
"""))